# Agent 5/6 â€” ESG Scoring & Climate

**What this notebook does:**  
Takes the raw ESG numbers from the provided datasets and converts them into clean, comparable scores (0â€“100) for E (Environmental), S (Social), G (Governance), and a combined ESG score. Pillar weights are sector-specific, calibrated to SASB materiality standards.

**Why we don't just use one vendor ESG score:**  
Academic research (Berg, Koelbel & Rigobon 2022 â€” *Aggregate Confusion*) shows that ESG scores from different vendors disagree significantly. So we build our own transparent score from raw variables â€” this way we can explain every number to the investor panel.

**Why SASB materiality?**  
Applying the same E/S/G weights to every company ignores sector context. SASB identifies which ESG factors are financially material per industry â€” we use these to set pillar weights before scoring.

**How to present this to investors:**  
> *Rather than relying on a black-box third-party ESG rating, we constructed our own transparent score from raw environmental, social, and governance variables. Pillar weights are calibrated to SASB materiality standards so that, for example, an energy company is judged primarily on its environmental performance while a bank is judged primarily on governance quality.*

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import glob

# Load master dataset from notebook 01
master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
if not master_files:
    raise FileNotFoundError("Run notebook 01 first to create the master dataset.")

master = pd.read_csv(master_files[-1])
print(f"Master dataset loaded: {len(master)} companies")
print("Columns available:", list(master.columns))

Master dataset loaded: 279 companies
Columns available: ['rc', 'idBbGlobal', 'idBbUnique', 'idBbCompany', 'idBbGlobalCompany', 'idBbGlobalCompanyName', 'idBbSecNumDes', 'ticker', 'exchCode', 'idIsin', 'idCusip', 'idSedol1', 'securityTyp', 'securityTyp2', 'classificationLevelCode1', 'classificationLevelName1', 'classificationLevelCode2', 'classificationLevelName2', 'classificationLevelCode3', 'classificationLevelName3', 'classificationLevelCode4', 'classificationLevelName4', 'classificationLevelCode5', 'classificationLevelName5', 'classificationLevelCode6', 'classificationLevelName6', 'classificationLevelCode7', 'classificationLevelName7', 'classificationScheme', 'rank', 'company', 'return_5y_pct', 'return_10y_pct', 'bb_ticker', 'yf_ticker', 'excluded', 'latestPeriodEndCsr', 'esgDisclosureScore', 'socialDisclosureScore', 'environDisclosureScore', 'directCo2Emissions', 'indirectCo2Emissions', 'totalCo2Emissions', 'totalGhgEmissions', 'scp3BusinessTravelEmissions', 'gasFlaring', 'emission

## Step 0 â€” SASB Materiality Weights

The Sustainability Accounting Standards Board (SASB) identifies which ESG factors are **financially material** for each industry sector. Rather than applying the same E/S/G pillar weights to every company, we adjust the weights based on the sector each company operates in.

**Why this matters:** A bank and an oil company should not be scored on the same ESG priorities. For an oil company, the Environmental pillar dominates. For a bank, Governance (data security, systemic risk, executive pay) is most material.

| Sector | E weight | S weight | G weight | Key SASB rationale |
|--------|----------|----------|----------|--------------------|
| Technology | 25% | 35% | 40% | Data security, employee engagement, governance |
| Financials | 20% | 35% | 45% | Systemic risk, data privacy, governance quality |
| Energy | 55% | 25% | 20% | GHG emissions, water, spills dominate |
| Consumer Discretionary | 35% | 40% | 25% | Supply chain labour, product safety |
| Consumer Staples | 35% | 40% | 25% | Supply chain, packaging, labour practices |
| Health Care | 25% | 40% | 35% | Patient safety, access, product quality |
| Industrials | 45% | 35% | 20% | GHG, occupational health, waste |
| Materials | 50% | 30% | 20% | Air/water pollution, hazardous waste |
| Real Estate | 45% | 25% | 30% | Energy efficiency, water management |
| Communication Services | 20% | 40% | 40% | Data privacy, content governance, diversity |
| Utilities | 50% | 25% | 25% | GHG intensity, renewable mix, water |
| Default (unmatched) | 40% | 30% | 30% | Balanced fallback |


In [2]:
# Create bics_sector alias from the Bloomberg classification column
if "bics_sector" not in master.columns:
    master["bics_sector"] = master["classificationLevelName1"]

# SASB materiality weights per BICS sector: (E_weight, S_weight, G_weight)
SASB_WEIGHTS = {
    "Technology":              (0.25, 0.35, 0.40),
    "Financials":              (0.20, 0.35, 0.45),
    "Energy":                  (0.55, 0.25, 0.20),
    "Consumer Discretionary":  (0.35, 0.40, 0.25),
    "Consumer Staples":        (0.35, 0.40, 0.25),
    "Health Care":             (0.25, 0.40, 0.35),
    "Industrials":             (0.45, 0.35, 0.20),
    "Materials":               (0.50, 0.30, 0.20),
    "Real Estate":             (0.45, 0.25, 0.30),
    "Communication Services":  (0.20, 0.40, 0.40),
    "Utilities":               (0.50, 0.25, 0.25),
}
DEFAULT_WEIGHTS = (0.40, 0.30, 0.30)

# Map each company to its sector weights
def get_sasb_weights(sector):
    for key in SASB_WEIGHTS:
        if isinstance(sector, str) and key.lower() in sector.lower():
            return SASB_WEIGHTS[key]
    return DEFAULT_WEIGHTS

master["sasb_e_weight"], master["sasb_s_weight"], master["sasb_g_weight"] = zip(
    *master["bics_sector"].apply(get_sasb_weights)
)

# Show the weight assigned to each sector present in the universe
weight_summary = (
    master[["bics_sector", "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"]]
    .drop_duplicates("bics_sector")
    .sort_values("bics_sector")
    .reset_index(drop=True)
)
print(f"Sectors in universe: {len(weight_summary)}")
print(weight_summary.to_string(index=False))

Sectors in universe: 11
           bics_sector  sasb_e_weight  sasb_s_weight  sasb_g_weight
        Communications           0.40           0.30           0.30
Consumer Discretionary           0.35           0.40           0.25
      Consumer Staples           0.35           0.40           0.25
                Energy           0.55           0.25           0.20
            Financials           0.20           0.35           0.45
           Health Care           0.25           0.40           0.35
           Industrials           0.45           0.35           0.20
             Materials           0.50           0.30           0.20
           Real Estate           0.45           0.25           0.30
            Technology           0.25           0.35           0.40
             Utilities           0.50           0.25           0.25


C:\Users\HP\AppData\Local\Temp\ipykernel_20432\2752441440.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master["bics_sector"] = master["classificationLevelName1"]
C:\Users\HP\AppData\Local\Temp\ipykernel_20432\2752441440.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master["sasb_e_weight"], master["sasb_s_weight"], master["sasb_g_weight"] = zip(
C:\Users\HP\AppData\Local\Temp\ipykernel_20432\2752441440.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` ma

## Step 1 â€” Choose which variables to include

After running the cell above, look at the column list and decide which columns belong to E, S, and G.

**Examples of typical columns:**
- **E (Environmental):** GHG emissions, Scope 1, Scope 2, carbon intensity, water usage, energy consumption
- **S (Social):** employee turnover, workplace accidents, gender pay gap, employee training hours
- **G (Governance):** board independence, board gender diversity, CEO pay ratio, audit committee independence

In [3]:
E_VARS = [
    "scope1_emissions_tco2e",
    "carbon_intensity_tco2e_per_eur_m_revenue",
    "renewable_energy_pct",
    "water_withdrawal_m3",
    "waste_recycled_pct",
]

S_VARS = [
    "women_in_workforce_pct",
    "gender_pay_gap_pct",
    "employee_turnover_pct",
    "work_related_injury_rate",
    "training_hours_per_employee",
]

G_VARS = [
    "board_independence_pct",
    "women_on_board_pct",
    "audit_committee_independence_pct",
    "executive_pay_esg_linked_pct",
    "ceo_pay_ratio",
]

# True = lower is better (e.g. emissions), False = higher is better
INVERT = {
    "scope1_emissions_tco2e":                  True,
    "carbon_intensity_tco2e_per_eur_m_revenue": True,
    "renewable_energy_pct":                    False,
    "water_withdrawal_m3":                     True,
    "waste_recycled_pct":                      False,
    "women_in_workforce_pct":                  False,
    "gender_pay_gap_pct":                      True,
    "employee_turnover_pct":                   True,
    "work_related_injury_rate":                True,
    "training_hours_per_employee":             False,
    "board_independence_pct":                  False,
    "women_on_board_pct":                      False,
    "audit_committee_independence_pct":        False,
    "executive_pay_esg_linked_pct":            False,
    "ceo_pay_ratio":                           True,
}

ALL_VARS = E_VARS + S_VARS + G_VARS
print(f"E variables: {len(E_VARS)}")
print(f"S variables: {len(S_VARS)}")
print(f"G variables: {len(G_VARS)}")

E variables: 5
S variables: 5
G variables: 5


## Step 2 â€” Normalise all variables to 0â€“100

We use min-max normalisation: the best company in each variable scores 100, the worst scores 0, everyone else is in between. This makes all variables comparable regardless of their units.

In [4]:
def normalise(series, invert=False):
    """Scale a series to 0-100. If invert=True, lower values get higher scores."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(50.0, index=series.index)  # all same value â†’ 50
    scaled = (series - mn) / (mx - mn) * 100
    return 100 - scaled if invert else scaled

scores = master[["ticker"] if "ticker" in master.columns else [master.columns[0]]].copy()

# Normalise each variable
for col in ALL_VARS:
    if col in master.columns:
        inv = INVERT.get(col, False)
        scores[f"{col}_score"] = normalise(master[col], invert=inv)
    else:
        print(f"WARNING: Column '{col}' not found in master dataset")

print("Variables normalised.")
scores.head(3)

Variables normalised.


,ticker,scope1_emissions_tco2e_score,carbon_intensity_tco2e_per_eur_m_revenue_score,renewable_energy_pct_score,water_withdrawal_m3_score,waste_recycled_pct_score,women_in_workforce_pct_score,gender_pay_gap_pct_score,employee_turnover_pct_score,work_related_injury_rate_score,training_hours_per_employee_score,board_independence_pct_score,women_on_board_pct_score,audit_committee_independence_pct_score,executive_pay_esg_linked_pct_score,ceo_pay_ratio_score
0,DP4B,71.881770,NaN,0.466497,NaN,NaN,28.415301,NaN,84.354839,NaN,NaN,53.846331,40.000000,100.000000,0.0,97.670185
1,SEBA,99.999999,NaN,0.110385,NaN,57.0,59.677214,NaN,82.258065,NaN,8.176215,37.062550,80.000000,39.999760,0.0,98.953693
2,ALV,99.978109,NaN,0.786808,99.994842,46.6,54.593976,99.99989,75.322581,96.074271,12.337591,42.307914,57.454545,51.999808,100.0,99.614202


## Step 3 â€” Calculate E, S, G and combined ESG scores

We weight each pillar equally (33%/33%/33%) unless you want to change this.  
The combined ESG score is a weighted average of E, S, and G.

In [5]:
def pillar_score(df, variables):
    cols = [f"{v}_score" for v in variables if f"{v}_score" in df.columns]
    if not cols:
        return pd.Series(np.nan, index=df.index)
    return df[cols].mean(axis=1)

scores["E_score"] = pillar_score(scores, E_VARS).round(2)
scores["S_score"] = pillar_score(scores, S_VARS).round(2)
scores["G_score"] = pillar_score(scores, G_VARS).round(2)

# Merge SASB weights into scores
scores = scores.merge(
    master[["ticker", "bics_sector", "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"]],
    on="ticker", how="left"
)

# Ensure numeric dtypes (pandas >= 2.1 no longer silently downcasts object arrays)
for col in ["E_score", "S_score", "G_score",
            "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"]:
    scores[col] = pd.to_numeric(scores[col], errors="coerce")

# Identify companies with NO pillar data at all
all_nan_mask = scores["E_score"].isna() & scores["S_score"].isna() & scores["G_score"].isna()
scores["esg_data_flag"] = "OK"
scores.loc[all_nan_mask, "esg_data_flag"] = "LOW_DATA"

scores["ESG_score"] = (
    scores["E_score"].fillna(scores["E_score"].median()) * scores["sasb_e_weight"] +
    scores["S_score"].fillna(scores["S_score"].median()) * scores["sasb_s_weight"] +
    scores["G_score"].fillna(scores["G_score"].median()) * scores["sasb_g_weight"]
).round(2).astype(float)
scores.loc[all_nan_mask, "ESG_score"] = np.nan

low_data_tickers = scores.loc[all_nan_mask, "ticker"].tolist()
print(f"LOW_DATA companies (all pillars NaN, excluded from portfolio): {low_data_tickers}")

final_scores = scores[[
    "ticker", "bics_sector",
    "E_score", "S_score", "G_score", "ESG_score", "esg_data_flag",
    "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"
]].copy()

# Ensure ESG_score is float for nlargest
final_scores["ESG_score"] = pd.to_numeric(final_scores["ESG_score"], errors="coerce")

print("\nTop 10 by ESG score (SASB-weighted):")
print(final_scores.nlargest(10, "ESG_score")[
    ["ticker", "bics_sector", "E_score", "S_score", "G_score",
     "sasb_e_weight", "sasb_s_weight", "sasb_g_weight", "ESG_score", "esg_data_flag"]
].to_string(index=False))

LOW_DATA companies (all pillars NaN, excluded from portfolio): ['FTD', '1ZG', '7CA', '2X1']

Top 10 by ESG score (SASB-weighted):
ticker            bics_sector  E_score  S_score  G_score  sasb_e_weight  sasb_s_weight  sasb_g_weight  ESG_score esg_data_flag
   NXG Consumer Discretionary    98.99    76.66    87.15           0.35           0.40           0.25      87.10            OK
  IHCB                 Energy   100.00    51.22    95.82           0.55           0.25           0.20      86.97            OK
   7MP             Financials   100.00    73.10    88.84           0.20           0.35           0.45      85.56            OK
  SKWA              Materials    95.11    61.19    78.57           0.50           0.30           0.20      81.63            OK
  EFEA            Real Estate   100.00    85.48    50.33           0.45           0.25           0.30      81.47            OK
  QS2A            Industrials    99.99    66.84    64.82           0.45           0.35           0.20      8

## Step 4 â€” ESG Triangulation

We cross-check our own ESG score against two external sources:
- **Bloomberg ESG Disclosure Score** â€” from the professor's provided CSV. Measures how completely a company discloses ESG data. Range 0â€“100, higher is better. Threshold: â‰¥ 50 = PASS.
- **Sustainalytics Risk Score** â€” fetched via yfinance (Yahoo Finance ESG tab). Measures unmanaged ESG risk. Range 0â€“100, **lower is better**. Threshold: â‰¤ 30 (medium risk or below) = PASS.

**Rule:** a company must pass at least **2 of 3** sources to clear triangulation. Companies that pass only 1 or fewer are flagged as **WATCHLIST** â€” they remain in the universe but are flagged for manual review before final selection.

| Source | Pass condition | Rationale |
|--------|---------------|----------|
| Agent score | â‰¥ 50 / 100 | Above-median in our own scoring |
| Bloomberg ESG Disclosure | â‰¥ 50 / 100 | Sufficient ESG transparency |
| Sustainalytics Risk | â‰¤ 30 / 100 | Medium risk or below |

In [6]:
import yfinance as yf
import time

# Source 2: Bloomberg ESG Disclosure Score
# Use idBbCompany (unique per company) to avoid ticker ambiguity
target_ids = set(master["idBbCompany"].dropna().astype(int).tolist())
esg_raw = pd.read_csv("../data/provided/esgEnvironmentalSocialConsolidatedV4.csv",
                      usecols=["idBbCompany", "esgDisclosureScore"])
esg_raw = esg_raw[esg_raw["idBbCompany"].isin(target_ids)].dropna(subset=["esgDisclosureScore"])
esg_raw = esg_raw.drop_duplicates("idBbCompany", keep="last")

id_to_ticker = master[["idBbCompany", "ticker"]].dropna().drop_duplicates("idBbCompany")
bloomberg_esg = id_to_ticker.merge(esg_raw, on="idBbCompany", how="left")
bloomberg_esg = bloomberg_esg[["ticker", "esgDisclosureScore"]].rename(
    columns={"esgDisclosureScore": "bloomberg_esg_disclosure"}
)
n_matched = bloomberg_esg["bloomberg_esg_disclosure"].notna().sum()
print(f"Bloomberg ESG disclosure matched: {n_matched} / {len(bloomberg_esg)} companies")

# Source 3: Sustainalytics via yfinance (use yf_ticker, not Bloomberg ticker)
yf_map = master[["ticker", "yf_ticker"]].dropna(subset=["yf_ticker"]).drop_duplicates("ticker")
print(f"Fetching Sustainalytics for {len(yf_map)} tickers...")

sust_rows = []
for i, row in enumerate(yf_map.itertuples(), 1):
    try:
        sust = yf.Ticker(row.yf_ticker).sustainability
        score = sust.loc["totalEsg", "Value"] if (sust is not None and "totalEsg" in sust.index) else None
    except Exception:
        score = None
    sust_rows.append({"ticker": row.ticker, "sustainalytics_risk_score": score})
    if i % 10 == 0:
        print(f"  {i}/{len(yf_map)} done...")
        time.sleep(1)

sust_df = pd.DataFrame(sust_rows)
n_sust = sust_df["sustainalytics_risk_score"].notna().sum()
print(f"Sustainalytics fetched: {n_sust} / {len(sust_df)} companies")

# Merge all three sources (all joins on Bloomberg ticker â€” 1:1 with final_scores)
tri = (
    final_scores[["ticker", "ESG_score"]]
    .merge(bloomberg_esg, on="ticker", how="left")
    .merge(sust_df,       on="ticker", how="left")
)

AGENT_THRESHOLD          = 50
BLOOMBERG_THRESHOLD      = 50
SUSTAINALYTICS_THRESHOLD = 30

tri["agent_pass"]          = tri["ESG_score"] >= AGENT_THRESHOLD
tri["bloomberg_pass"]      = tri["bloomberg_esg_disclosure"] >= BLOOMBERG_THRESHOLD
tri["sustainalytics_pass"] = tri["sustainalytics_risk_score"] <= SUSTAINALYTICS_THRESHOLD

tri["tri_sources_available"] = (
    tri["ESG_score"].notna().astype(int) +
    tri["bloomberg_esg_disclosure"].notna().astype(int) +
    tri["sustainalytics_risk_score"].notna().astype(int)
)
tri["tri_passes"] = (
    tri["agent_pass"].fillna(False).astype(int) +
    tri["bloomberg_pass"].fillna(False).astype(int) +
    tri["sustainalytics_pass"].fillna(False).astype(int)
)

def triangulation_result(row):
    if row["tri_sources_available"] < 2:
        return "LOW_DATA"
    return "PASS" if row["tri_passes"] >= 2 else "WATCHLIST"

tri["triangulation_result"] = tri.apply(triangulation_result, axis=1)

final_scores = final_scores.merge(
    tri[["ticker", "bloomberg_esg_disclosure", "sustainalytics_risk_score",
         "tri_sources_available", "tri_passes", "triangulation_result"]],
    on="ticker", how="left"
)

print("Triangulation results:")
print(final_scores["triangulation_result"].value_counts().to_string())
print()
watchlist = final_scores[final_scores["triangulation_result"] == "WATCHLIST"]
print(f"WATCHLIST ({len(watchlist)} companies):")
print(watchlist[["ticker", "ESG_score", "bloomberg_esg_disclosure",
                  "sustainalytics_risk_score", "tri_passes"]].to_string(index=False))


Bloomberg ESG disclosure matched: 275 / 279 companies
Fetching Sustainalytics for 279 tickers...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: MAERSK-B.CO"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SEB-A.ST"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: ALV.DE"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: RXL.PA"}}}


...

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: RILBA.CO"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SHELL.AS"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SBNOR.OL"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: TBCG.L"}}}


Sustainalytics fetched: 0 / 279 companies
Triangulation results:
triangulation_result
PASS         165
WATCHLIST    110
LOW_DATA       4

WATCHLIST (110 companies):
ticker  ESG_score  bloomberg_esg_disclosure sustainalytics_risk_score  tri_passes
  DP4B      47.67                     58.41                      None           1
   GBF      52.76                     40.84                      None           1
  TCO0      62.39                     48.12                      None           1
  RTO1      66.66                     48.12                      None           1
   BSI      51.79                     44.28                      None           1
   DTE      44.33                     56.00                      None           1
   8RJ      42.31                     50.84                      None           1
  SUL1      69.31                     46.10                      None           1
   IVS      62.09                     38.26                      None           1
  NCYD      64.

## Step 5 â€” WACI (Weighted Average Carbon Intensity)

WACI is the standard carbon metric required by the assignment.  
It measures how carbon-intensive the portfolio is, weighted by how much we invest in each company.  
Unit: tonnes of COâ‚‚ per million euros of revenue (tCOâ‚‚e / â‚¬m revenue)

In [7]:
# Carbon intensity: use Bloomberg Calc value where available,
# fall back to sector median for missing companies.
# Bloomberg co2IntensityPerSalesCalc is an estimated/modelled field.
# Imputed values are flagged ci_source = sector_median_imputed.

CARBON_COL   = "carbon_intensity_tco2e_per_eur_m_revenue"
SECTOR_COL_W = "classificationLevelName1"

ci = master[["ticker", CARBON_COL, SECTOR_COL_W]].drop_duplicates("ticker").copy()
ci.columns = ["ticker", "carbon_intensity", "sector"]

# Compute sector medians from reported values
sector_medians = ci.dropna(subset=["carbon_intensity"]).groupby("sector")["carbon_intensity"].median()
global_median  = ci["carbon_intensity"].median()

print("Sector median carbon intensity (tCO2e/EUR m revenue):")
for sector, median in sector_medians.sort_values(ascending=False).items():
    n_reported = ci[ci["sector"] == sector]["carbon_intensity"].notna().sum()
    n_total    = (ci["sector"] == sector).sum()
    print(f"  {sector:<30} {median:>8.1f}  ({n_reported}/{n_total} reported)")

# Impute missing values with sector median (or global if sector has no data)
def impute_ci(row):
    if pd.notna(row["carbon_intensity"]):
        return row["carbon_intensity"], "bloomberg_calc"
    fallback = sector_medians.get(row["sector"], global_median)
    return fallback, "sector_median_imputed"

ci[["carbon_intensity", "ci_source"]] = ci.apply(
    lambda r: pd.Series(impute_ci(r)), axis=1
)

n_reported = (ci["ci_source"] == "bloomberg_calc").sum()
n_imputed  = (ci["ci_source"] == "sector_median_imputed").sum()
print(f"\nCoverage: {n_reported} reported, {n_imputed} sector-median imputed")
print(f"\nTop 10 highest carbon intensity:")
print(ci.sort_values("carbon_intensity", ascending=False)[["ticker","carbon_intensity","ci_source"]].head(10).to_string(index=False))


Sector median carbon intensity (tCO2e/EUR m revenue):
  Materials                        4134.0  (4/23 reported)
  Utilities                         629.8  (1/18 reported)
  Energy                            144.7  (4/15 reported)
  Consumer Staples                   85.4  (1/8 reported)
  Health Care                        15.7  (4/20 reported)
  Industrials                         9.8  (8/60 reported)
  Financials                          0.7  (7/94 reported)
  Consumer Discretionary              0.0  (1/12 reported)

Coverage: 30 reported, 249 sector-median imputed

Top 10 highest carbon intensity:


ticker  carbon_intensity             ci_source
   UCM           6439.03        bloomberg_calc
  HLBN           4285.61        bloomberg_calc
  NOH1           4134.00 sector_median_imputed
  KGHA           4134.00 sector_median_imputed
   AIL           4134.00 sector_median_imputed
  VACD           4134.00 sector_median_imputed
  HL9C           4134.00 sector_median_imputed
   SDF           4134.00 sector_median_imputed
   NDA           4134.00 sector_median_imputed
   RTZ           4134.00 sector_median_imputed


## Step 6 â€” Save ESG scores

In [8]:
today = str(date.today())

# Merge carbon intensity + imputation source into ESG scores
if "carbon_intensity" in ci.columns:
    final_scores = final_scores.merge(
        ci[["ticker", "carbon_intensity", "ci_source"]], on="ticker", how="left"
    )

final_scores["data_vintage"] = today
out_path = f"../outputs/scores/esg_scores_{today}.csv"
final_scores.to_csv(out_path, index=False)
print(f"ESG scores saved: {out_path}")
print(f"Columns: {list(final_scores.columns)}")
print("\nScore summary:")
final_scores[["E_score","S_score","G_score","ESG_score"]].describe().round(2)


ESG scores saved: ../outputs/scores/esg_scores_2026-05-20.csv
Columns: ['ticker', 'bics_sector', 'E_score', 'S_score', 'G_score', 'ESG_score', 'esg_data_flag', 'sasb_e_weight', 'sasb_s_weight', 'sasb_g_weight', 'bloomberg_esg_disclosure', 'sustainalytics_risk_score', 'tri_sources_available', 'tri_passes', 'triangulation_result', 'carbon_intensity', 'ci_source', 'data_vintage']

Score summary:


,E_score,S_score,G_score,ESG_score
count,256.00,264.00,275.00,275.00
mean,72.47,53.01,66.12,63.68
std,19.47,16.76,15.76,9.23
min,0.00,0.00,0.00,32.40
25%,58.22,43.60,55.43,58.03
50%,67.79,53.74,65.34,63.72
75%,93.18,64.45,75.78,69.79
max,100.00,92.18,99.98,87.10


## Notebook complete

You now have:
- **SASB materiality weights** per sector
- **E, S, G and combined ESG scores** (0â€“100), sector-weighted
- **ESG Triangulation** â€” Bloomberg disclosure + Sustainalytics cross-check, 2-of-3 rule
- **WATCHLIST flags** for companies that fail triangulation
- **Carbon intensity** per company

Output files:
- `outputs/scores/esg_scores_DATE.csv` â€” full ESG scores with triangulation columns

**Next:** Open `agent11_portfolio_construction.ipynb` to select the final 15â€“25 holdings.

---

## Step 7 — Import FactSet ESG data from ESG Specialist

Replaces the broken in-house triangulation (Bloomberg Disclosure + broken yfinance Sustainalytics)
with the specialist's proper FactSet output: **Truvalue + Sustainalytics + ISS** (2-of-3 rule).

**Source files** (in `data/provided/`):
- `Portfolio_Screening_Output.xlsx` — Stage 1 vendor screening (289 companies), hard exclusions list, Stage 2 full ranking with E/S/G pillar z-scores
- `stage2_top40_capped_hybrid.csv` — sector-capped top 40 (reference / sanity check)

**Output:** saves a new `esg_scores_{today}.csv` which NB10 automatically picks up (it globs for the latest file).

**How to run:** execute the cell below independently — you do NOT need to re-run the full NB05 pipeline first. The cell reads the specialist's files and the current master dataset, then writes the new ESG scores file.

In [9]:
import pandas as pd
import numpy as np
import re
import glob
from datetime import date

TODAY = str(date.today())
PORTFOLIO_FILE = "../data/provided/Portfolio_Screening_Output.xlsx"

# ── 1. Load specialist sheets ─────────────────────────────────────────────────
s1_all    = pd.read_excel(PORTFOLIO_FILE, sheet_name="Stage 1 — Universe")
hard_raw  = pd.read_excel(PORTFOLIO_FILE, sheet_name="Hard exclusions detail")
s2_full   = pd.read_excel(PORTFOLIO_FILE, sheet_name="Stage 2 — Full ranking")
hard_excl_names = set(hard_raw["Company"].str.strip().str.lower())

print(f"Stage 1 universe: {len(s1_all)} companies")
print(f"Hard exclusions:  {len(hard_raw)} companies")
print(f"Stage 2 ranked:   {len(s2_full)} companies")

# ── 2. Load master dataset for ticker mapping ─────────────────────────────────
master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
master = pd.read_csv(master_files[-1], usecols=["ticker", "yf_ticker", "idBbGlobalCompanyName",
                                                  "carbon_intensity_tco2e_per_eur_m_revenue",
                                                  "classificationLevelName1"])
master = master.drop_duplicates("ticker")
print(f"Master dataset:   {len(master)} companies")

# ── 3. Company name normalisation ─────────────────────────────────────────────
_SUFFIXES = sorted([
    "class b", "class a", "class c", "ab", "sa", "nv", "plc", "ag", "asa",
    "oyj abp", "oyj", "oy", "ltd", "inc", "corp", "spa", "s p a", "s a",
    "se", "a s", "as", "group", "holding", "holdings", "sca", "psc",
], key=len, reverse=True)

def norm(s):
    s = re.sub(r"[^\w\s]", " ", str(s).lower())
    s = re.sub(r"\s+", " ", s).strip()
    for sfx in _SUFFIXES:
        if s.endswith(" " + sfx):
            s = s[: -len(sfx) - 1].strip()
            break
    return s

master["_key"]  = master["idBbGlobalCompanyName"].apply(norm)
s1_all["_key"]  = s1_all["Company"].apply(norm)
s2_full["_key"] = s2_full["Company"].apply(norm)

# ── 4. Join Stage 1 universe → master ticker ──────────────────────────────────
merged = s1_all.merge(
    master[["_key", "ticker", "yf_ticker", "carbon_intensity_tco2e_per_eur_m_revenue",
            "classificationLevelName1"]],
    on="_key", how="left"
)

n_matched = merged["ticker"].notna().sum()
unmatched = merged.loc[merged["ticker"].isna(), "Company"].tolist()
print(f"\nMatched {n_matched}/{len(merged)} specialist companies to master ticker")
if unmatched:
    print(f"Unmatched (not in current master — no financial data yet): {unmatched[:10]}")

# ── 5. Add Stage 2 ranking scores ─────────────────────────────────────────────
s2_scores = s2_full[["_key", "Rank", "In-house ESG z", "Percentile",
                      "E pillar", "S pillar", "G pillar",
                      "Watchlist", "Watchlist reason"]].rename(columns={
    "Rank": "esg_rank", "In-house ESG z": "in_house_z",
    "Percentile": "in_house_pct",
    "E pillar": "E_z", "S pillar": "S_z", "G pillar": "G_z",
    "Watchlist": "watchlist", "Watchlist reason": "watchlist_reason",
})
merged = merged.merge(s2_scores, on="_key", how="left")

# ── 6. Normalise E/S/G pillar z-scores → 0–100 ───────────────────────────────
for z_col, out_col in [("E_z", "E_score"), ("S_z", "S_score"), ("G_z", "G_score")]:
    z = merged[z_col]
    z_min, z_max = z.min(), z.max()
    if z_max > z_min:
        merged[out_col] = ((z - z_min) / (z_max - z_min) * 100).clip(0, 100).round(2)
    else:
        merged[out_col] = 50.0
    merged[out_col] = merged[out_col].fillna(0.0)

# ── 7. Hard-excluded flag and final ESG_score ─────────────────────────────────
merged["hard_excluded"]  = merged["Company"].str.strip().str.lower().isin(hard_excl_names)
merged["fossil_flag"]    = merged["Fossil flag"].fillna(False).astype(bool)

# ESG_score = in_house_pct (0–100 percentile) for Stage 2 companies
# Stage 1 failures and hard exclusions get 0 — they fall below ESG_FLOOR in NB10
merged["ESG_score"] = np.where(
    merged["Stage 1 PASS"] & ~merged["hard_excluded"],
    merged["in_house_pct"].fillna(0.0),
    0.0
).round(2)

# ── 8. Carbon intensity: use master value, fall back to sector median ─────────
ci_col = "carbon_intensity_tco2e_per_eur_m_revenue"
sector_median = (
    merged.dropna(subset=[ci_col])
    .groupby("classificationLevelName1")[ci_col]
    .median()
)
global_median = merged[ci_col].median()

def get_ci(row):
    if pd.notna(row[ci_col]):
        return row[ci_col], "bloomberg_calc"
    fallback = sector_median.get(row["classificationLevelName1"], global_median)
    return (fallback if pd.notna(fallback) else global_median), "sector_median_imputed"

merged[["carbon_intensity", "ci_source"]] = merged.apply(
    lambda r: pd.Series(get_ci(r)), axis=1
)

# ── 9. Assemble output columns ────────────────────────────────────────────────
out = merged[[
    "ticker", "yf_ticker", "Company", "SASB Sector",
    "Truvalue", "SA Risk", "SA Band", "ISS",
    "PASS votes", "FAIL votes", "Stage 1 vendor verdict", "Stage 1 PASS",
    "fossil_flag", "hard_excluded", "watchlist", "watchlist_reason",
    "esg_rank", "in_house_z", "in_house_pct",
    "E_score", "S_score", "G_score", "ESG_score",
    "carbon_intensity", "ci_source",
]].rename(columns={
    "Company":                 "company_name",
    "SASB Sector":             "sasb_sector",
    "Truvalue":                "truvalue_rating",
    "SA Risk":                 "sa_risk",
    "SA Band":                 "sa_band",
    "ISS":                     "iss_descriptor",
    "PASS votes":              "tri_pass_votes",
    "FAIL votes":              "tri_fail_votes",
    "Stage 1 vendor verdict":  "triangulation_result",
    "Stage 1 PASS":            "stage1_pass",
})

# Only keep rows where we have a project ticker (i.e., company is in master)
out_matched = out[out["ticker"].notna()].copy()
out_matched["data_vintage"] = TODAY

# ── 10. Save ──────────────────────────────────────────────────────────────────
out_path = f"../outputs/scores/esg_scores_{TODAY}.csv"
out_matched.to_csv(out_path, index=False)

print(f"\n✓ Saved: {out_path}")
print(f"  Companies with ticker match: {len(out_matched)}")
print(f"  Stage 1 PASS:               {out_matched['stage1_pass'].sum()}")
print(f"  Hard excluded:              {out_matched['hard_excluded'].sum()}")
print(f"  Fossil flagged:             {out_matched['fossil_flag'].sum()}")
print(f"  Watchlist:                  {out_matched['watchlist'].sum()}")
print(f"\nESG_score distribution (matched companies):")
print(out_matched["ESG_score"].describe().round(2))
print(f"\nTop 15 by ESG_score:")
print(out_matched[["company_name", "ESG_score", "triangulation_result", "watchlist", "truvalue_rating"]]
      .sort_values("ESG_score", ascending=False)
      .head(15)
      .to_string(index=False))

Stage 1 universe: 289 companies
Hard exclusions:  13 companies
Stage 2 ranked:   224 companies
Master dataset:   279 companies

Matched 213/289 specialist companies to master ticker
Unmatched (not in current master — no financial data yet): ['A.P. Moller - Maersk A/S Class B', 'BE Semiconductor Industries N.V.', 'ABN AMRO Bank N.V. Depositary receipts', 'Addtech AB Class B', 'ALK-abello A/S Class B', 'ASM International N.V.', 'ASR Nederland N.V.', 'Gaztransport & Technigaz', 'Logitech', 'BELIMO']

✓ Saved: ../outputs/scores/esg_scores_2026-05-20.csv
  Companies with ticker match: 213
  Stage 1 PASS:               179
  Hard excluded:              11
  Fossil flagged:             17
  Watchlist:                  24

ESG_score distribution (matched companies):
count    213.00
mean      42.99
std       33.03
min        0.00
25%        6.33
50%       43.04
75%       71.31
max      100.00
Name: ESG_score, dtype: float64

Top 15 by ESG_score:
                    company_name  ESG_score trian